# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PundarikakshNTripathi/FlyRankAI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Random Forest Classifier**

Why? Random Forests handle non-linear interactions natively (e.g., age matters differently depending on baseline impressions and position). They are robust to outliers and provide straightforward feature importances.

## 2. Split design

**Group Shuffle Split by Client**

We split by `client_id` to ensure the model generalizes to completely unseen domains. A random split would leak domain-specific URL structures or formatting quirks into the training set.

## 3. Train + compare vs my baseline

Training the Random Forest and comparing its Precision@50 against the baseline.

In [5]:
import os
data_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/content_refresh_anonymized.csv'
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(data_path)
features = ['impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'word_count', 'ctr', 'avg_position', 'engagement_rate']
for f in features:
    df[f] = df[f].fillna(0)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

X_train, y_train = df.loc[train_idx, features], df.loc[train_idx, 'is_declining']
X_test, y_test = df.loc[test_idx, features], df.loc[test_idx, 'is_declining']

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# Evaluate Precision@50
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rf_scores = rf.predict_proba(X_test)[:, 1]

# Baseline score on test set
stale_visible = ((X_test['days_since_last_update'] >= 180) & (X_test['impressions_90d'] >= 500)).astype(int)
page_1_decay = ((X_test['avg_position'] > 0) & (X_test['avg_position'] <= 10) & (X_test['content_age_days'] >= 180)).astype(int)
baseline_scores = stale_visible + page_1_decay

print(f'Baseline P@50: {precision_at_k(baseline_scores, y_test, 50):.3f}')
print(f'Random Forest P@50: {precision_at_k(rf_scores, y_test, 50):.3f}')


Baseline P@50: 0.560
Random Forest P@50: 0.700


## 4. Errors and interpretation

The Random Forest outperforms the static baseline significantly at the top of the queue. Feature importance shows that historical `impressions_90d` and `content_age_days` are the dominant drivers of the split.

In [6]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances.head(5))


impressions_90d           0.331661
content_age_days          0.210519
avg_position              0.208785
word_count                0.097936
days_since_last_update    0.057434
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.